# İzge: devam ettirilebilir Colab eğitimi
Bu notebook burada çalıştırılmadı. Yalnız onaylı, insan etiketli seçilmiş görüntüleri kullanın. Ham video yüklemeyin. Mevcut gerçek insan etiketi sayısı sıfırdır; bu hazırlık eğitim başarısı değildir.

Colab dosya panelinden kod paketini kendiniz `/content/izge` altına açın. Paket `uv.lock`, `pyproject.toml`, `backend/` ve `config/training/approved/` içermelidir. Runner ayrı `infra/training-runner` işinden gelir; bu notebook runner uygulamaz. Python 3.14 ve ffprobe gerekir; Colab sistem Python'unu değiştirmeden uv ortamı kurulur.

In [ ]:
from pathlib import Path
import os
import subprocess
REPO = Path('/content/izge')
MOUNT_DRIVE = False  # Yalnız kullanıcı bu değeri değiştirip hücreyi çalıştırınca kimlik doğrulanır.
INSTALL = False
RUN_TRAINING = False
RUN_NAME = 'human-approved-run'
CONFIG = REPO / 'config/training/approved/CHANGE_ME.yaml'
DATA_ROOT = Path('/content/drive/MyDrive/izge/datasets/approved')
MANIFEST = DATA_ROOT / 'manifests/posture/v1/manifest.json'
# Notebook bağımlılığı sabit; eğitim bağımlılıkları depodaki uv.lock ile --frozen kurulur.
if INSTALL:
    subprocess.run(['pip', 'install', 'uv==0.13.5'], check=True)
    subprocess.run(['uv', 'sync', '--frozen', '--extra', 'gpu'], cwd=REPO, check=True)

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive mount kapalı; kullanıcı açık eylemi gerekir.')

## Veri kapısı
Onaylı config, bu veri kökünü ve yalnız train/val girdilerini kullanan manifest okuyucusunu seçmelidir. Manifestin `load_manifest` API'sini kullanın; JSON'u doğrudan okuyan eğitim adaptörü bu kapıyı sağlamaz. Test dosyalarını ayrı yerel pakette tutun; Drive'a eğitim için taşımayın. `--final-eval` bu notebook'ta kullanılmaz. Dosya sistemi izinleriyle mühürleme ayrıca veri sahibinin sorumluluğudur.

Veri pipeline manifesti insan etiketi veya öğrenme onayı değildir. COCO için mevcut `export_behavior_annotations.py --validate-export` kapısını, duruş için `POSTURE_TRAINING_GATE.md` kapılarını ayrıca tamamlayın. Onaylı config ve runner burada yoksa devam etmeyin.

In [ ]:
assert DATA_ROOT.is_dir() and REPO.is_dir()
assert CONFIG.is_file() and CONFIG.resolve().parent == (REPO / 'config/training/approved').resolve()
env = dict(os.environ, PYTHONPATH=str(REPO / 'backend'), IZGE_DATA_ROOT=str(DATA_ROOT))
subprocess.run(['uv', 'run', '--frozen', 'python', '-m', 'izge.datasets',
                'check-manifest', str(MANIFEST), '--data-root', str(DATA_ROOT)],
               cwd=REPO, env=env, check=True)
# Ayrı eğitim/etiket kabulü tamamlandıktan sonra kullanıcı açıkça değiştirir.
HUMAN_DATA_APPROVED = False

In [ ]:
assert RUN_TRAINING and HUMAN_DATA_APPROVED, 'Gerçek veri ve config onayı bekleniyor'
assert RUN_NAME and '/' not in RUN_NAME and RUN_NAME not in {'.', '..'}
RUN_DIR = Path('/content/drive/MyDrive/izge/runs') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
env['MLFLOW_TRACKING_URI'] = (RUN_DIR / 'mlflow').as_uri()
subprocess.run(['uv', 'run', '--frozen', 'python', '-m', 'izge.training.run',
                '--config', str(CONFIG), '--run-dir', str(RUN_DIR), '--resume'],
               cwd=REPO, env=env, check=True)

## Bağlantı koparsa
Aynı kod/uv.lock, config, veri manifesti ve `RUN_NAME` ile yeni runtime açın. Kurulum ve kullanıcı Drive mount hücrelerini yeniden çalıştırın. Veri kapısını tekrar geçin, ardından aynı komutu `--resume` ile çalıştırın. `status.json`, `metrics.jsonl`, `checkpoints/last.pt` ve MLflow dosyaları aynı Drive run dizininde kalır. Başka bir çalışma aynı dizine yazmamalı. Checkpoint bulunmuyorsa kurtarma mümkün olduğunu varsaymayın; runner durumunu inceleyin. Config/veri değiştiyse yeni run adı kullanın.

Colab GPU, Drive kimlik doğrulaması, runner eğitimi ve runtime kesintisi kurtarması bu ortamda doğrulanmadı.